# W11 Practice — Dijkstra and A* on a Grid
**EE0849 Introduction to Robotics**

In this practice session you'll implement the **two graph-search algorithms** from the W11 slides:

1. **Dijkstra** — explores cells in order of cheapest cost from the start.
2. **A\*** — same idea, but uses a heuristic to prefer cells that look closer to the goal.

You'll only write the algorithm code. The grid, neighbor lookup, and plotting are already done — just run those cells.


In [ ]:
# Run once on Colab / fresh envs
!pip install numpy matplotlib --quiet


## Setup — just run these cells
The next cell defines:
- `make_grid()` → a small test grid with walls
- `neighbors(grid, cell)` → yields `(neighbor, step)` pairs (cardinal = 1, diagonal = √2), with no corner-cutting
- `octile(a, b)` → the heuristic from the slides
- `visualize(...)` → plots the grid, expanded cells, and final path

You **don't need to read or modify** the helpers — just run the cell.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from math import sqrt

SQRT2 = sqrt(2)

def make_grid():
    """Returns (grid, start, goal). 1 = obstacle, 0 = free."""
    grid = np.zeros((10, 15), dtype=int)
    grid[3, 2:9]   = 1   # horizontal wall
    grid[3:7, 8]   = 1   # short vertical wall
    grid[6, 9:13]  = 1   # second horizontal wall
    return grid, (8, 1), (1, 13)

def neighbors(grid, cell):
    """Yield ((ni, nj), step) for free 8-connected neighbors.
       Diagonal step is √2; cardinal step is 1.
       No corner-cutting: skip diagonal if EITHER adjacent cardinal is a wall."""
    rows, cols = grid.shape
    i, j = cell
    moves = [(-1, 0, 1), (1, 0, 1), (0, -1, 1), (0, 1, 1),
             (-1, -1, SQRT2), (-1, 1, SQRT2), (1, -1, SQRT2), (1, 1, SQRT2)]
    for di, dj, step in moves:
        ni, nj = i + di, j + dj
        if not (0 <= ni < rows and 0 <= nj < cols): continue
        if grid[ni, nj] == 1: continue
        if di != 0 and dj != 0 and (grid[i, nj] == 1 or grid[ni, j] == 1): continue
        yield (ni, nj), step

def octile(a, b):
    """Octile heuristic — exact 8-connected distance on an obstacle-free grid."""
    di, dj = abs(a[0] - b[0]), abs(a[1] - b[1])
    return SQRT2 * min(di, dj) + abs(di - dj)

def reconstruct_path(parent, goal):
    """Walk parent pointers from goal back to start."""
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    return path[::-1]

def visualize(grid, expanded=None, path=None, start=None, goal=None, title=''):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(8, 5))
    # Walls
    for i in range(rows):
        for j in range(cols):
            if grid[i, j] == 1:
                ax.add_patch(Rectangle((j, rows - 1 - i), 1, 1,
                                       facecolor='#1e293b', edgecolor='#0f172a'))
    # Expanded (visited)
    if expanded:
        for i, j in expanded:
            ax.add_patch(Rectangle((j, rows - 1 - i), 1, 1,
                                   facecolor='#cbd5e1', edgecolor='#94a3b8', alpha=0.7))
    # Grid lines
    for k in range(cols + 1):
        ax.plot([k, k], [0, rows], color='#e2e8f0', lw=0.5, zorder=0)
    for k in range(rows + 1):
        ax.plot([0, cols], [k, k], color='#e2e8f0', lw=0.5, zorder=0)
    # Path
    if path:
        xs = [c + 0.5 for _, c in path]
        ys = [rows - 0.5 - r for r, _ in path]
        ax.plot(xs, ys, color='#7c3aed', lw=3, marker='o', ms=5, zorder=4)
    # Start / goal
    if start:
        ax.add_patch(Rectangle((start[1], rows - 1 - start[0]), 1, 1,
                               facecolor='#22c55e', edgecolor='#15803d', lw=2))
        ax.text(start[1] + 0.5, rows - 0.5 - start[0], 'S',
                ha='center', va='center', color='white', fontsize=14, fontweight='bold')
    if goal:
        ax.add_patch(Rectangle((goal[1], rows - 1 - goal[0]), 1, 1,
                               facecolor='#f39c12', edgecolor='#b45309', lw=2))
        ax.text(goal[1] + 0.5, rows - 0.5 - goal[0], 'G',
                ha='center', va='center', color='white', fontsize=14, fontweight='bold')
    ax.set_xlim(0, cols); ax.set_ylim(0, rows); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    expanded_count = len(expanded) if expanded else 0
    cost_str = f', cost = {sum(SQRT2 if a[0]!=b[0] and a[1]!=b[1] else 1 for a,b in zip(path, path[1:])):.2f}' if path else ''
    ax.set_title(f'{title} — {expanded_count} cells expanded{cost_str}',
                 fontsize=12, color='#1c5b84', fontweight='bold')
    plt.show()

# Build the test grid
grid, start, goal = make_grid()
visualize(grid, start=start, goal=goal, title='Test grid')


---
## Exercise 1 — Implement Dijkstra

Fill in the three TODOs below.

**The algorithm**:
1. Track `cost[cell]` (cheapest known cost from start) and `parent[cell]` (which cell we came from).
2. Repeatedly pick the unvisited cell with the **smallest cost**, mark it visited, and look at its neighbors.
3. For each `(neighbor, step)`: if `cost[current] + step` is cheaper than `cost[neighbor]`, update it.
4. Stop when you visit `goal`.

`open_set` is the set of cells discovered but not yet visited (the **frontier**).


In [ ]:
def dijkstra(grid, start, goal):
    INF = float('inf')
    cost     = {start: 0}
    parent   = {start: None}
    open_set = {start}
    expanded_order = []   # for visualization

    while open_set:
        # === TODO 1: pick the cell with the smallest cost in open_set ===
        current = ...

        open_set.remove(current)
        expanded_order.append(current)

        if current == goal:
            return reconstruct_path(parent, goal), expanded_order

        # === TODO 2: for each (neighbor, step) in neighbors(grid, current):
        #            compute new_cost; if cheaper than the best we have, ===
        #            update cost, parent, and add to open_set.
        for neighbor, step in neighbors(grid, current):
            ...

    return None, expanded_order


# Test it
path, expanded = dijkstra(grid, start, goal)
visualize(grid, expanded=expanded, path=path, start=start, goal=goal, title='Dijkstra')


---
## Exercise 2 — Implement A\*

The **only difference** from Dijkstra: when picking the next cell, sort by `f = g + h` instead of just `g`.

- `g[cell]` = best cost from start to this cell (same as Dijkstra's `cost`)
- `h[cell]` = heuristic estimate from this cell to goal — use the provided `octile(cell, goal)`
- `f = g + h`

Copy your Dijkstra code below and change **just the picking step**.


In [ ]:
def astar(grid, start, goal):
    INF = float('inf')
    cost     = {start: 0}
    parent   = {start: None}
    open_set = {start}
    expanded_order = []

    while open_set:
        # === TODO: pick the cell with the smallest f = g + h ===
        # hint: f(c) = cost[c] + octile(c, goal)
        current = ...

        open_set.remove(current)
        expanded_order.append(current)

        if current == goal:
            return reconstruct_path(parent, goal), expanded_order

        for neighbor, step in neighbors(grid, current):
            new_cost = cost[current] + step
            if new_cost < cost.get(neighbor, INF):
                cost[neighbor] = new_cost
                parent[neighbor] = current
                open_set.add(neighbor)

    return None, expanded_order


path, expanded = astar(grid, start, goal)
visualize(grid, expanded=expanded, path=path, start=start, goal=goal, title='A*')


---
## Exercise 3 — Compare Dijkstra and A\*

Run both side-by-side. Notice:
- Both should find the **same path cost** (A\* with an admissible heuristic is optimal).
- A\* should expand **fewer cells** because the heuristic biases it toward the goal.


In [ ]:
path_d, exp_d = dijkstra(grid, start, goal)
path_a, exp_a = astar(grid, start, goal)

print(f'Dijkstra: {len(exp_d)} cells expanded')
print(f'A*:       {len(exp_a)} cells expanded')
print(f'A* expanded {100 * (1 - len(exp_a)/len(exp_d)):.0f}% fewer cells')

visualize(grid, expanded=exp_d, path=path_d, start=start, goal=goal, title='Dijkstra')
visualize(grid, expanded=exp_a, path=path_a, start=start, goal=goal, title='A*')


---
## Bonus — Try a different heuristic

Replace `octile` in your A\* with one of the heuristics from the slides:

```python
def manhattan(a, b):  return abs(a[0]-b[0]) + abs(a[1]-b[1])      # inadmissible on 8-conn
def chebyshev(a, b):  return max(abs(a[0]-b[0]), abs(a[1]-b[1]))  # admissible, loose
def euclidean(a, b):  return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5   # admissible, loose
def zero(a, b):       return 0   # this turns A* back into Dijkstra
```

Plug each into A\*, count expansions, and check whether the path length stays optimal (8.66… for our grid). Which heuristic gives the fewest expansions while still finding the optimal path?
